## Apresentação 1 - Cálculo Numérico
# A REDE HIDRÁULICA
**Grupo 1:** Bruno Lopes de Almeida Zuffo, 
João Victor Bozola Bosi, 
Lucas Antero Albuquerque, 
Matheus Marchi Baron, 
Natalia Yumi Watanabe, 
Victor Hugo Albertino e Silva.

---

## 1. Introdução e Construção da Rede

Nossa modelagem traduz a geometria física da rede hidráulica para a **Teoria dos Grafos**.
As equações de conservação de massa são representadas através de uma matriz global de condutâncias. A grande vantagem numérica desta abordagem é que a montagem da estrutura espacial escala de forma previsível (linearmente).

In [1]:
%load_ext autoreload
%autoreload 2

import matplotlib
matplotlib.use("TkAgg")
import time
import numpy as np
import matplotlib.pyplot as plt

from functions import Assembly, GeraGrafo, PlotaRede, SolveNetwork, createK, createD, calc_vazao, calc_potencia, AssemblyVectorC

# CONSTRUÇÃO DO MODELO FÍSICO
Xno, conec = GeraGrafo(levels=3)
mm_to_m = 0.001
Xno = Xno * mm_to_m
C = AssemblyVectorC(conec, Xno)
n_inlet = 0                 # nó mais a esquerda
n_outlet = len(Xno) - 1     # nó mais a direita

print(f"Rede gerada com {Xno.shape[0]} nós e {conec.shape[0]} canos.\n")

Rede gerada com 224 nós e 327 canos.



---

## 2. O Estado Estacionário (Itens 1, 2 e 3)

Para cenários onde a vazão de injeção externa é conhecida (Itens 1 e 2), a resolução é direta através do sistema $\tilde{A}\mathbf{p} = \mathbf{b}$. Bastou algumas modificações na função `SolveNetwork` para usar dicionários como os parâmetros para os valores das pressões e vazões. 

No **Item 3**, o desafio é o inverso dos anteriores: nós não sabemos a vazão da bomba, mas sabemos que ela precisa ter força suficiente para cravar a pressão do Inlet em exatamente **100 Pa** (Condição de Dirichlet). 

Para descobrir a vazão exata de forma otimizada, exploramos a **linearidade do sistema** e o Princípio da Superposição.

Por ser um sistema de equações lineares estático, a matriz geométrica $\tilde{A}$ obedece às propriedades distributiva e de homogeneidade escalar. Se temos duas soluções conhecidas $\tilde{A}\mathbf{p}^{(1)} = \mathbf{b}^{(1)}$ e $\tilde{A}\mathbf{p}^{(2)} = \mathbf{b}^{(2)}$, ao aplicarmos uma combinação linear $\mathbf{p} = \alpha \mathbf{p}^{(1)} + \beta \mathbf{p}^{(2)}$, temos:

$$\tilde{A}(\alpha \mathbf{p}^{(1)} + \beta \mathbf{p}^{(2)}) = \tilde{A}(\alpha \mathbf{p}^{(1)}) + \tilde{A}(\beta \mathbf{p}^{(2)})$$

Como os escalares $\alpha$ e $\beta$ podem ser fatorados da multiplicação matricial:

$$= \alpha(\tilde{A}\mathbf{p}^{(1)}) + \beta(\tilde{A}\mathbf{p}^{(2)})$$

Substituindo pelas igualdades iniciais, provamos que a resposta a uma soma de estímulos proporcionais é a soma proporcional das respostas individuais:

$$\tilde{A}\mathbf{p} = \alpha \mathbf{b}^{(1)} + \beta \mathbf{b}^{(2)}$$

Para resolver o Item 3, aplicamos a equação provada acima usando apenas um estado base (logo, assumimos $\beta = 0$ e trabalhamos apenas com $\alpha$ e $\mathbf{b}^{(1)}$). 

1. Injetamos um vetor de vazão de teste $\mathbf{b}^{(1)}$ (**1.0e-6 m³/s**) e resolvemos o sistema para descobrir o campo de pressões $\mathbf{p}^{(1)}$ (a nossa $p_{teste}$).
2. Como queremos que a pressão no nó de entrada atinja nosso alvo de **100 Pa**, calculamos o nosso multiplicador escalar $\alpha$ isolando a variável:
$$\alpha = \frac{p_{alvo}}{p_{teste}}$$
3. Pela linearidade comprovada, o vetor de vazões reais ($\mathbf{b}$) necessário para o problema é simplesmente a nossa vazão de teste escalonada pelo multiplicador $\alpha$:
$$\mathbf{b} = \alpha \mathbf{b}^{(1)}$$

In [ ]:
# A: Itens 1 e 2

ps_A = {
    '6': 0.0,       # nó 5 -> pressão atmosférica (saída 1)
    '216': 0.0      # nó 215 -> pressão atmosférica (saída 2)
}

Qs_A = {
    '1': 1.0e-7,    # nó 0 -> injeção de vazão 0.1 mL/s
    '176': 1.0e-6    # nó 175 -> injeção de vazão 1 mL/s
}

pressure_A = SolveNetwork(conec, C, ps=ps_A, Qs=Qs_A)
matriz_vazao_A = calc_vazao(conec, C, pressure_A)

# B: Item 3

ps_B = {str(n_outlet + 1): 0}

# vazão arbitrária de teste (b^(1))
Q_teste = 1.0e-6
Qs_B = {'1': Q_teste}

# pressão gerada por essa vazão de teste (p^(1))
pressure_teste = SolveNetwork(conec, C, ps=ps_B, Qs=Qs_B)
pressao_no_inlet = pressure_teste[0] # pressão no nó de entrada

# escalar alpha para atingir os 100 Pa desejados
pressao_alvo = 100.0
alpha = pressao_alvo / pressao_no_inlet

vazao_inlet_real = alpha * Q_teste

print(f"Escalar alpha = {alpha:.2f}:")
print(f"Vazão necessária para manter 100 Pa no Inlet: {vazao_inlet_real:.4e} m³/s\n")

Escalar alpha = 0.04):
Vazão necessária para manter 100 Pa no Inlet: 3.7797e-08 m³/s



---

## 3. Dinâmica Linear no Tempo (Itens 4 e 5)

Neste cenário, a bomba não injeta mais uma vazão constante, mas sim oscilante, descrita por ondas de $\sin(\omega t)$ e $\cos(\omega t)$. Além disso, exploramos a injeção em pontos diferentes da placa:
* **Item 4:** Injeção acompanhando a função $\sin$ no **Nó 0** (Inlet original).
* **Item 5:** Injeção acompanhando a função $\cos$ no **Nó 175** (No meio da placa).

**Superposição Linear**:
Se fôssemos pedir para o computador recalcular o sistema linear inteiro ($\tilde{A}p = b$) para cada milissegundo dessa oscilação, a simulação demoraria muito. Mas nós não precisamos fazer isso.

Como a água não esquenta nessa etapa, a física da placa continua a mesma. A nossa matriz $A$ é totalmente constante ao longo do tempo. A matemática nos garante que:
$$A \cdot p(t) = Q(t)$$

Se a vazão injetada $Q(t)$ oscila no tempo acompanhando uma onda — por exemplo, $Q(t) = Q_{base} \cdot (1 + 0.1 \sin(\omega t))$ —, podemos usar a proporcionalidade a nosso favor. 

Substituindo essa função na nossa equação principal e aproveitando a propriedade linear do sistema matricial, temos:
$$A \cdot p(t) = Q_{base} \cdot (1 + 0.1 \sin(\omega t))$$

Sabendo que a pressão de base é dada por $A \cdot p_{base} = Q_{base}$, deduzimos que a pressão no tempo $t$ é simplesmente:

$$p(t) = p_{base} \cdot (1 + 0.1 \sin(\omega t))$$

Nós resolvemos a matriz "pesada" do sistema apenas **uma única vez** no pré-processamento para descobrir qual é a pressão de base ($p_{base}$).

In [ ]:
# C: Itens 4 e 5
t_array = np.linspace(0, 10, 1000) 
omega_sin = 3.0 
omega_cos = 4.0

ps_base = {str(n_outlet + 1): 0}

# Solução Base - Nó 0 (Item 4)
Qs_base_sin = {'1': 1.0e-6} 
pressao_base_sin = SolveNetwork(conec, C, ps=ps_base, Qs=Qs_base_sin)
pressao_maxima_base_sin = np.max(pressao_base_sin)

# Solução Base - Nó 175 (Item 5)
Qs_base_cos = {'176': 1.0e-6} 
pressao_base_cos = SolveNetwork(conec, C, ps=ps_base, Qs=Qs_base_cos)
pressao_maxima_base_cos = np.max(pressao_base_cos)

pressao_maxima_tempo_sin = []
pressao_maxima_tempo_cos = []

# Superposição
for t in t_array:
    fator_tempo_sin = 1.0 + 0.1 * np.sin(omega_sin * t)
    fator_tempo_cos = 0.1 + 0.01 * np.cos(omega_cos * t)
    
    p_max_atual_sin = pressao_maxima_base_sin * fator_tempo_sin
    p_max_atual_cos = pressao_maxima_base_cos * fator_tempo_cos
    
    pressao_maxima_tempo_sin.append(p_max_atual_sin)
    pressao_maxima_tempo_cos.append(p_max_atual_cos)

# Plot
plt.figure(figsize=(10, 6))
plt.plot(t_array, pressao_maxima_tempo_sin, color='blue', linewidth=2, label=f'Nó 0: Injeção com sin({omega_sin}t)')
plt.plot(t_array, pressao_maxima_tempo_cos, color='orange', linewidth=2, label=f'Nó 175: Injeção com cos({omega_cos}t)')

plt.title("Pressão Máxima com Injeção Dinâmica (Itens 4 e 5)")
plt.xlabel("Tempo (s)")
plt.ylabel("Pressão Máxima na Rede (Pa)")
plt.grid(True, linestyle=':', alpha=0.7)
plt.legend(loc="upper right")
plt.show()

---

## 4. O Efeito Termofluidodinâmico (Item 6)

O cenário muda quando o fluido sofre aquecimento. A temperatura aumenta quadraticamente com o tempo e dita a viscosidade dinâmica através da seguinte relação:

$$\mu(T) = \frac{0.001791}{1 + 0.03368T + 0.000221T^2}$$

**A Quebra da Linearidade:** Diferente dos Itens 4 e 5, o sistema não é mais invariante no tempo. Como a viscosidade altera a facilidade de escoamento da água, a matriz de condutâncias inteira ($\tilde{A}$) se deforma fisicamente a cada instante. A superposição linear não funciona mais aqui; somos forçados a refazer a resolução linear pesada a cada passo de tempo.

**Otimização Algébrica (A Dedução):**<br>
Para evitar o recálculo custoso de toda a geometria a cada instante $t$, isolamos a parcela geométrica da equação de condutância.

A fórmula geral da condutância de um microcanal é:
$$C_k = \frac{\pi D_k^4}{128 \mu L_k}$$

No estado inicial (água a 20 °C), a viscosidade é $\mu = 0.001$ Pa·s. Logo, a condutância original armazenada na memória é:
$$C_{original} = \frac{\pi D_k^4}{128 \cdot 0.001 \cdot L_k}$$

Isolando os parâmetros geométricos (que são constantes e não dilatam no modelo):
$$\frac{\pi D_k^4}{128 L_k} = C_{original} \cdot 0.001$$

Para um instante de tempo $t$, a nova condutância depende exclusivamente da nova viscosidade $\mu(t)$. Separando a parte geométrica da parte fluida, temos:
$$C_{novo}(t) = \left( \frac{\pi D_k^4}{128 L_k} \right) \cdot \frac{1}{\mu(t)}$$

Substituindo a constante geométrica isolada no passo anterior, chegamos à equação final que utilizamos no código para atualizar a rede inteira com uma única operação escalar:
$$C_{novo}(t) = (C_{original} \cdot 0.001) \cdot \frac{1}{\mu(t)} \Rightarrow$$
$$\Rightarrow \boxed{C_{novo}(t) = C_{original} \cdot \left( \frac{0.001}{\mu(t)} \right)}$$

In [ ]:
# D: Item 6
t_array = np.linspace(0,10,100) 

ps_D = {str(n_outlet + 1): 0}
Qs_D = {'1': 1.0e-7} 

max_pressure = [] 

for t in t_array:
    T_t = 20 + 0.9*(t**2)
    mu_t = 0.001791 / (1 + 0.03368 * T_t + 0.000221 * (T_t**2))

    C_t = C * (0.001/mu_t)

    pressure_t = SolveNetwork(conec, C_t, ps=ps_D, Qs=Qs_D)
    max_pressure.append(np.max(pressure_t))

plt.figure(figsize=(10, 5))
plt.plot(t_array, max_pressure, color='red', linewidth=2)
plt.title("Efeito do Aquecimento na Pressão Máxima (Item 6)")
plt.xlabel("Tempo (s)")
plt.ylabel("Pressão Máxima na Rede (Pa)")
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

---

## 5. Análise de Desempenho Computacional (Item 7)

Na engenharia computacional, é fundamental avaliar a viabilidade e a escalabilidade do código. Isolamos o tempo de **Montagem do Sistema** do tempo de **Resolução (Escalonamento Matricial)** utilizando um cronômetro de alta precisão do Python com média de 10 execuções, filtrando interferências do SO.

Os resultados provam numericamente que, enquanto o pré-processamento geométrico escala de forma comportada, o motor de álgebra linear exibe sua complexidade $\mathcal{O}(N^3)$, tornando-se o verdadeiro gargalo de desempenho à medida que a rede se aproxima do Nível 7.

In [ ]:
# E: Item 7
print(f"{'-' * 62}\n{'Nível':<7} | {'Qtd. Nós':<10} | {'T. Montagem (s)':<18} | {'T. Resolução (s)':<18}\n{'-' * 62}")

for level in [1, 2, 3, 4]:
    Xno_test, conec_test = GeraGrafo(levels=level)
    Xno_test = Xno_test * mm_to_m
    n_nos = Xno_test.shape[0]

    tempos_montagem = []
    tempos_resolucao = []

    for _ in range(10):
        
        # MONTAGEM
        t_inicio_montagem = time.perf_counter()
        C_test = AssemblyVectorC(conec_test, Xno_test)
        A_test = Assembly(conec_test, C_test)
        t_fim_montagem = time.perf_counter()
        tempos_montagem.append(t_fim_montagem - t_inicio_montagem)

        # RESOLUÇÃO        
        Atilde_test = A_test.copy()
        idx_out = n_nos - 1 
        Atilde_test[idx_out, :] = 0
        Atilde_test[idx_out, idx_out] = 1
        b_test = np.zeros(n_nos) 
        b_test[0] = 1.0e-7 

        t_inicio_resolucao = time.perf_counter()
        pressure_test = np.linalg.solve(Atilde_test, b_test)
        t_fim_resolucao = time.perf_counter()
        tempos_resolucao.append(t_fim_resolucao - t_inicio_resolucao)

    media_montagem = np.mean(tempos_montagem)
    media_resolucao = np.mean(tempos_resolucao)
    
    print(f"{level:<7} | {n_nos:<10} | {media_montagem:<18.6f} | {media_resolucao:<18.6f}\n")


--------------------------------------------------------------
Nível   | Qtd. Nós   | T. Montagem (s)    | T. Resolução (s)  
--------------------------------------------------------------
1       | 38         | 0.000236           | 0.000033          

2       | 92         | 0.000560           | 0.000067          

3       | 224        | 0.001446           | 0.000394          

4       | 490        | 0.003945           | 0.003085          



---
# EXTRA: Detalhando o `functions.py`

Nesta seção, apresentamos a lógica matemática das funções que compõem o núcleo do nosso Gêmeo Digital. O foco está na transformação de dados geométricos em um sistema de equações lineares e no pós-processamento dos resultados.
### 1.1 Funções de Geometria e Condutância
O primeiro passo é transformar a lista de conectividade e as coordenadas em propriedades físicas.
* **`AssemblyVectorC`**: Calcula o comprimento $L_k$ de cada cano e utiliza a função **`CalculoCondutancia`** para montar o vetor de condutâncias.
* **`CalculoCondutancia`**: Implementa a física de microcanais. A condutância $C_k$ é calculada pela lei de Poiseuille:
$$C_k = \frac{\pi D_k^4}{128 \mu L_k}$$

In [ ]:
def CalculoCondutancia(Lk):
    pi = np.pi
    Ak = 2.5e-7
    mi = 0.001 # Viscosidade a 20°C
    Dk = np.sqrt(4*Ak/pi)
    kk = (pi * Dk**4) / (128 * mi)
    return kk / Lk

def AssemblyVectorC(conec, Xno):
    nc = len(conec)
    C = np.zeros(nc)
    for k in range(nc):
        n1, n2 = conec[k, 0], conec[k, 1]
        x1, y1 = Xno[n1, 0], Xno[n1, 1]
        x2, y2 = Xno[n2, 0], Xno[n2, 1]
        Lk = np.sqrt((x2 - x1)**2 + (y2 - y1)**2)
        C[k] = CalculoCondutancia(Lk)
    return C

### 1.2 Montagem da Matriz Global (`Assembly`)
Esta função constrói a matriz de rigidez do sistema $A$. Ela aplica a conservação de massa (Lei de Kirchhoff) em cada nó. Para cada cano $k$ ligando $n_1$ e $n_2$:
* Soma-se $C_k$ nas posições $(n_1, n_1)$ e $(n_2, n_2)$ (auto-influência).
* Subtrai-se $C_k$ nas posições $(n_1, n_2)$ e $(n_2, n_1)$ (influência mútua).

In [ ]:
def Assembly(conec, C):
    conec = np.array(conec)
    nv = int(conec.max() + 1)
    nc = len(conec)
    A = np.zeros((nv, nv))
    for k in range(nc):
        n1, n2 = conec[k, 0], conec[k, 1]
        A[n1, n1] += C[k]
        A[n1, n2] -= C[k]
        A[n2, n1] -= C[k]
        A[n2, n2] += C[k]
    return A

### 1.3 O Solver (`SolveNetwork`)
Esta função prepara o sistema $\tilde{A}p = b$. 
* **Vazões ($Qs$)**: São adicionadas diretamente ao vetor $b$ (Condição de Neumann).
* **Pressões ($ps$)**: Aplica-se a condição de Dirichlet. A linha da matriz é zerada, o termo da diagonal vira $1$ e o valor da pressão vai para o vetor $b$.

In [ ]:
def SolveNetwork(conec, C, ps=None, Qs=None):
    A = Assembly(conec, C)
    Atilde = A.copy()
    n = A.shape[0]
    b = np.zeros(n)

    if Qs is not None:
        for node, value in Qs.items():
            b[int(node) - 1] = value

    if ps is not None:
        for node, value in ps.items():
            i = int(node) - 1
            Atilde[i, :] = 0
            Atilde[i, i] = 1
            b[i] = value

    return np.linalg.solve(Atilde, b)

### 1.4 Operadores Lineares e Leis de Formação (`createK` e `createD`)

Para extrair vazões e potências de forma eficiente, estruturamos a topologia e a física da rede em operadores matriciais.

#### A Matriz de Condutância ($K$)
É uma matriz diagonal de dimensão $(nc \times nc)$. Sua lei de formação define que cada cano $i$ possui sua própria facilidade de fluxo, sem interferência direta entre as condutâncias:
$$K_{i,j} = \begin{cases} C_i & \text{se } i = j \\ 0 & \text{se } i \neq j \end{cases}$$

#### A Matriz de Incidência ($D$)
É uma matriz de dimensão $(nc \times nv)$ que mapeia a conectividade. Para cada cano $k$ que conecta um nó de entrada $n_1$ a um nó de saída $n_2$, a lei de formação é:
$$D_{k,j} = \begin{cases} 1 & \text{se } j = n_1 \\ -1 & \text{se } j = n_2 \\ 0 & \text{caso contrário} \end{cases}$$

**A Lei de Ohm Hidráulica Matricial:**
A composição desses operadores permite calcular a vazão em todos os canos simultaneamente:
$$Q = K \cdot D \cdot p$$
Onde o produto $D \cdot p$ resulta no vetor de diferenças de pressão $\Delta p$ em cada trecho.

In [ ]:
def createK(C, nc):
    K = np.zeros((nc, nc))
    for i in range(nc):
        for j in range(nc):
            if i == j:
                K[i][j] = C[i] # Elemento da diagonal principal
    return K

def createD(conec, nv, nc):
    D = np.zeros((nc, nv))
    for k in range(nc):
        n1 = conec[k, 0] # Nó de origem do cano k
        n2 = conec[k, 1] # Nó de destino do cano k
        for j in range(nv):
            if j == n1:
                D[k][j] = 1
            elif j == n2:
                D[k][j] = -1
    return D

### 1.5 Pós-Processamento (`calc_vazao` e `calc_potencia`)
Após obter as pressões, resgatamos as grandezas de fluxo.
* **Vazão ($Q$)**: Calculada pela relação matricial $Q = K D p$.
* **Potência ($W$)**: Calculada pela forma quadrática $W = p^T (D^T K D) p$, representando a energia dissipada viscosamente.

In [ ]:
def calc_vazao(conec, C, pressure):
    nv, nc = int(conec.max() + 1), len(conec)
    K = createK(C, nc)
    D = createD(conec, nv, nc)
    return K @ D @ pressure

def calc_potencia(conec, C, pressure):
    nv, nc = int(conec.max() + 1), len(conec)
    K = createK(C, nc)
    D = createD(conec, nv, nc)
    return pressure.T @ (D.T @ K @ D) @ pressure